In [ ]:
########################################
#getting system arguments
import sys
def GetArg_dataName(default="Variables"):
    """
    #non-perturbations
    #Run One: python Tracked_Profiles.py Variables
    #Run Two: python Tracked_Profiles.py Entrainment
    #Run Three: python Tracked_Profiles.py PROCESSED_Entrainment (also can add _DivideMassFlux)
    #Run Four: python Tracked_Profiles.py W_Budgets
    #Run Five: python Tracked_Profiles.py QV_Budgets
    #Run Six: python Tracked_Profiles.py TH_Budgets

    #perturbations
    #Run One: python Tracked_Profiles.py Variables_Perturbations
    """
    # If run inside Jupyter, sys.argv will include ipykernel arguments
    if any("ipykernel_launcher" in arg for arg in sys.argv):
        print(f"Using default dataName: {default}")
        return default

    # If a user-specified argument exists, use it
    if len(sys.argv) > 1:
        out=sys.argv[1]
        print(f"Using argument dataName: {out}")
        return out

    return default

dataName = GetArg_dataName()

In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm
import copy
import warnings

from matplotlib.colors import LogNorm
# from scipy.interpolate import interp1d    

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import *

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=1)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="Tracking_Algorithms", dataName="Lagrangian_UpdraftTracking",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#data manager class (for saving data)
DataManager_TrackedProfiles = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="Tracked_Profiles", dataName="Tracked_Ascent_Trajectories",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","2_Tracking_Algorithms"))
from CLASSES_TrackingAlgorithms import TrackingAlgorithms_DataLoading_Class, Results_InputOutput_Class, TrackedParcel_Loading_Class

In [ ]:
# IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","3_Tracked_Profiles"))
from CLASSES_TrackedProfiles import TrackedProfiles_DataLoading_CLASS

In [ ]:
#IMPORT FUNCTIONS

import sys
path=os.path.join(mainCodeDirectory,'Functions/')
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

In [ ]:
##############################################
#DATA LOADING FUNCTIONS

In [ ]:
def GetData_V1(trackedArray):
    pIndices=trackedArray[:,0]
    
    varNames=GetVarNames()
    
    dataDictionary = {varName: np.full((len(ModelData.timeStrings), len(pIndices)), np.nan) for varName in varNames}
    
    for t in tqdm(range(len(ModelData.timeStrings)), position=2, leave=False):
        for varName in varNames:
            if any(prefix in var for var in varNames for prefix in ("E_")) and t == ModelData.Ntime-1:
                dataDictionary[varName][t] = np.full((1, len(pIndices)), np.nan)
            else:
                ###############
                dataDictionary[varName][t] = CallLagrangianArray(ModelData, DataManager, \
                                                                 ModelData.timeStrings[t], varName)[pIndices]
                ###############

    return dataDictionary

def GetData_V2(trackedArray):
    pIndices=trackedArray[:,0]
    
    varNames=GetVarNames()
    dataDictionary = {varName: np.full((len(ModelData.timeStrings), len(pIndices)), np.nan) for varName in varNames}
    
    for t in tqdm(range(len(ModelData.timeStrings)), position=2, leave=False):
        dataLocationInfoCache = {'inputDataDirectory': None, 'dataName': None}; currentFile = None
        
        for varName in varNames:
            if any(prefix in var for var in varNames for prefix in ("E_")) and t == ModelData.Ntime-1:
                dataDictionary[varName][t] = np.full((1, len(pIndices)), np.nan)
            else:
                ###############
                [inputDataDirectory,dataName] = CallLagrangianArray(ModelData, DataManager, \
                                                                    ModelData.timeStrings[t], \
                                                                    varName,loadData=False) #get file info
                if inputDataDirectory != dataLocationInfoCache['inputDataDirectory']\
                or dataName != dataLocationInfoCache['dataName']: #if not in cache, open new file
                    # print('reloading',varName)
                    if currentFile is not None: currentFile.close() #close the old file
                    currentFile=DataManager.GetTimestepData_V2(inputDataDirectory, ModelData.timeStrings[t], dataName)
                    dataLocationInfoCache['inputDataDirectory']=inputDataDirectory #set cache
                    dataLocationInfoCache['dataName']=dataName #set cache
                dataDictionary[varName][t] = currentFile[varName][:][pIndices] #grab the needed variable
                ###############
        if currentFile is not None: currentFile.close() #close the file at end of each timestep

    return dataDictionary

def GetVarNames():
    varNames = ['z','LFC','QCQI']
    if dataName == 'Variables':
        varNames += ['W','QV','THETA_v','THETA_e','BUOYANCY2'] #... ['HMC']
    elif "Entrainment" in dataName:
        varNames += ['PROCESSED_E_DivideMassFlux_c','PROCESSED_D_DivideMassFlux_c'] #*testing
    return varNames

In [ ]:
##############################################
#COMPUTING FUNCTIONS

In [ ]:
def RunCode(trackedArrays,parcelTypes):

    parcelDepths=['SHALLOW','DEEP','ALL']
    parcelDepths=['ALL'] #testing
    
    #subset out parcelType and parcelDepth
    trajectoriesArrayDictionary = {pt: {pd: {} for pd in parcelDepths} for pt in parcelTypes}
    priorToAscentArrayDictionary = {pt: {pd: {} for pd in parcelDepths} for pt in parcelTypes} #*
    for parcelType in tqdm(parcelTypes, position=0, leave=True):
        for parcelDepth in tqdm(parcelDepths, desc=f"Type: {parcelType}", position=1, leave=False):

            #subsetting data
            trackedArray = trackedArrays[parcelType][parcelDepth]
            t1 = trackedArray[:, 1].astype(int); t2 = trackedArray[:, 2].astype(int)
            after = trackedArray[:, 3].astype(int)
        
            # Data Calculations
            # loading variable data dictionary
            tqdm.write(f"Getting Data")
            dataDictionary = GetData_V2(trackedArray)
            
            #getting data
            varNames=GetVarNames()
            for varName in tqdm(varNames, desc="Subsetting Variable", leave=False):
                variableData = dataDictionary[varName]
                
                #computing trajectories for each variable
                trajectoriesArray = SubsetTrajectories(variableData,t1,t2,after)
                trajectoriesArrayDictionary[parcelType][parcelDepth][varName] = trajectoriesArray
                
                priorToAscentArray = SubsetTrajectories(variableData,t1=0,t2=t1,after=0) #*
                priorToAscentArrayDictionary[parcelType][parcelDepth][varName] = priorToAscentArray #*

    return trajectoriesArrayDictionary,priorToAscentArrayDictionary

def SubsetTrajectories(variableData,t1,t2,after):
    #making output array
    trajectoriesArray = np.full_like(variableData,fill_value=np.nan,dtype=float)

    #masking the final output array
    time_idx = np.arange(variableData.shape[0])[:, np.newaxis]
    mask = (time_idx >= t1) & (time_idx <= t2+after)
    
    #applying mask to output array
    trajectoriesArray[mask] = variableData[mask]

    return trajectoriesArray

def LimitTrackedArraysRows(trackedArrays, limit=None): #limit=(0,10000)
    if limit is None:
        return trackedArrays
    for parcelType in trackedArrays:
        for parcelDepth in trackedArrays[parcelType]:
            trackedArrays[parcelType][parcelDepth] \
            = trackedArrays[parcelType][parcelDepth][limit[0]:limit[1], :]
    return trackedArrays

# Np=107076; Nt=661; Nvars=5;
# Nparceltypes=2
# Np*Nt*Nvars*Nparceltypes*4/1e9

In [ ]:
##############################################
#COMPUTING
running=True #KEEP TRUE WHEN JOBARRAY IS RUNNING
running=False 

In [ ]:
if running:
    #Loading in Tracked Parcels Info
    trackedArrays,_ = TrackedParcel_Loading_Class.LoadingSubsetParcelData(ModelData,DataManager,Results_InputOutput_Class)
    trackedArrays=LimitTrackedArraysRows(trackedArrays) #limiting number of parcels to allow computation to complete

    # parcelTypesList = [['CL','nonCL']]
    # for parcelTypes in parcelTypesList:
    #     [trajectoriesArrayDictionary,priorToAscentArrayDictionary] = RunCode(trackedArrays,parcelTypes)
    #     TrackedProfiles_DataLoading_CLASS.SaveProfile(ModelData,DataManager_TrackedProfiles,trajectoriesArrayDictionary, dataName, t=f'trajectoriesArray_{parcelTypes[0]}vs{parcelTypes[1]}')
    #     TrackedProfiles_DataLoading_CLASS.SaveProfile(ModelData,DataManager_TrackedProfiles,priorToAscentArrayDictionary, dataName, t=f'priorToAscent_{parcelTypes[0]}vs{parcelTypes[1]}')

In [ ]:
##############################################
#POST-PROCESSING

In [ ]:
#Loading Data
if not running:
    parcelTypesList = [['CL','nonCL']]; parcelTypes = parcelTypesList[0]
    trajectoriesArrayDictionary = TrackedProfiles_DataLoading_CLASS.LoadProfile(ModelData,DataManager_TrackedProfiles, dataName, t=f'trajectoriesArray_{parcelTypes[0]}vs{parcelTypes[1]}')
    priorToAscentArrayDictionary = TrackedProfiles_DataLoading_CLASS.LoadProfile(ModelData,DataManager_TrackedProfiles, dataName, t=f'priorToAscent_{parcelTypes[0]}vs{parcelTypes[1]}')

In [ ]:
##############################################
#HISTOGRAM CALCULATION FUNCTIONS

In [ ]:
#Calculation Functions
def BinData(PrepareData_Function,
            dataDictionary,varName='W',relative='LFC'):    
    _,_,_, time_bins,z_bins = PrepareData_Function(dataDictionary, 0, varName=varName,relative=relative)
    counts_all = np.zeros((len(time_bins)-1, len(z_bins)-1))
    weights_all = np.zeros((len(time_bins)-1, len(z_bins)-1))
    
    for p in range(dataDictionary['z'].shape[1]):
        time_relative,z_relative,data, _,_ = PrepareData_Function(dataDictionary, p, varName=varName,relative=relative)
        
        c, xedges, yedges = np.histogram2d(time_relative, z_relative, bins=[time_bins, z_bins])
        w, _, _ = np.histogram2d(time_relative, z_relative, bins=[xedges, yedges], weights=data)
        counts_all += c
        weights_all += w
    
    meanData = np.where(counts_all > 0, weights_all / counts_all, np.nan)
    return meanData, time_bins,z_bins

def MakeCalculations(PrepareData_Function,arrayDictionary,
                     varNames,parcelTypes,parcelDepth='SHALLOW',relative='LFC'):
    meanDataDictionary = {}
    for varName in tqdm(varNames):
        meanDataDictionary[varName] = {}
        for parcelType in parcelTypes:
            dataDictionary = arrayDictionary[parcelType][parcelDepth]
            meanData, time_bins, z_bins = BinData(PrepareData_Function,
                                                  dataDictionary,varName=varName, relative=relative)
            meanDataDictionary[varName][parcelType] = meanData    
    return meanDataDictionary, time_bins,z_bins

def CombineMeanData(meanDataDictionary_before, time_bins_before,
                    meanDataDictionary_after,  time_bins_after):
    
    # combined time bins: drop last edge of before (=0) to avoid duplicate
    time_bins_combined = np.concatenate([time_bins_before[:-1], time_bins_after])

    meanDataDictionary_combined = {}
    for varName in meanDataDictionary_before:
        meanDataDictionary_combined[varName] = {}
        for parcelType in meanDataDictionary_before[varName]:
            before = meanDataDictionary_before[varName][parcelType]  # (Nt_before, Nz)
            after  = meanDataDictionary_after[varName][parcelType]   # (Nt_after,  Nz)
            meanDataDictionary_combined[varName][parcelType] = np.concatenate([before, after], axis=0)

    return meanDataDictionary_combined, time_bins_combined

In [ ]:
# Calculating Functions (Up Until First BL Acceleration)
def PrepareData_UpUntilFirstBLAcceleration(dataDictionary,p=0,varName='W',relative=None):
    #getting
    z = dataDictionary['z'][:,p]
    qcqi = dataDictionary['QCQI'][:,p]
    LFC = dataDictionary['LFC'][:,p]
    data = dataDictionary[varName][:,p]
    time = (np.arange(ModelData.Ntime)*ModelData.dt/3600+6)*60 

    mask = np.isfinite(z)
    
    #masking
    time = time[mask]; time_relative = time-time[-1]; z=z[mask]
    data=data[mask]
    z_relative = z #just original z
    
    z_bins = np.arange(0, 6000+25, 25)
    time_bins = np.arange(-30, 0+1, int(ModelData.dt/60))
    return time_relative,z_relative,data, time_bins,z_bins

# Calculating Functions (First BL Acceleration to Cloud Exit)
def PrepareData_FirstBLAccelerationToCloudExit(dataDictionary,p=0,varName='W',relative='LFC'):
    #getting
    z = dataDictionary['z'][:,p]
    qcqi = dataDictionary['QCQI'][:,p]
    LFC = dataDictionary['LFC'][:,p]
    data = dataDictionary[varName][:,p]
    time = (np.arange(ModelData.Ntime)*ModelData.dt/3600+6)*60 

    mask = np.isfinite(z)
    
    #masking
    time = time[mask]; time_relative = time-time[0]; z=z[mask]
    data=data[mask]
    if relative=='CB':
        cloudMask = qcqi[mask] > 1e-6
        if not np.any(cloudMask):
            time_relative[:] = np.nan
            z_relative = np.full_like(z, np.nan)
            data[:] = np.nan
        else: 
            CB = z[cloudMask][0]
            z_relative = z - CB    
        z_bins = np.arange(-3000, 6000+50, 50)
    elif relative=='LFC':
        LFC=LFC[mask]; z_relative = z-LFC
        z_bins = np.arange(-3000, 6000+50, 50)
    elif relative is None:
        z_relative = z
        z_bins = np.arange(0, 6000+25, 25)
    
    time_bins = np.arange(0, 60+1, int(ModelData.dt/60))
    return time_relative,z_relative,data, time_bins,z_bins

# #testing
# p=3
# [time_relative,z_relative,data, time_bins,z_bins] = PrepareData(dataDictionary,p,varName='W')
# plt.plot(time_relative,z_relative)
# plt.ylabel('z - z_LFC (km)'); plt.xlabel('time since ascent (mins)')
# plt.axhline(0, color='grey', zorder=-10, linestyle='--',label='LFC')

In [ ]:
##############################################
#PLOTTING FUNCTIONS

In [ ]:
#Plotting Functions
def PlotBinnedData(meanData_1, meanData_2, time_bins, z_bins,
                   varName,
                   # interpolate=False, 
                   multiplier=1, cmap='RdBu_r', symmetric=True,
                   label='',
                   zLabel='z - z_LFC (m)',
                   titles=['Panel 1', 'Panel 2'],
                   xLabel='time since right before ascent (mins)',
                   plotType='contourf',numLevels=21):

    fig, axes = plt.subplots(3, 1, figsize=(8, 10), sharex=True, sharey=True)
    #set background color
    if plotType == 'contourf':
        for axis in axes.ravel():
            axis.set_facecolor('grey')

    #get vmin_threshold
    vmin_threshold = 1e-6 if varName == "QCQI" else None
    
    #organize data
    meanDatas = [meanData_1, meanData_2]
    #apply multiplier
    meanDatas = [multiplier * meanData for meanData in meanDatas]
    if vmin_threshold is not None: vmin_threshold*=multiplier

    # #interpolate
    # if interpolate: meanDatas = [InterpolateBinnedData(meanData,time_bins) for meanData in meanDatas]

    #Coordinates
    if plotType == 'pcolormesh':
        plotX = time_bins[:-1]; plotY = z_bins[:-1]
    elif plotType == 'contourf':
        plotX = 0.5 * (time_bins[:-1] + time_bins[1:]); plotY = 0.5 * (z_bins[:-1] + z_bins[1:])

    #single plots colorbar setup
    if vmin_threshold is not None:
        meanDatas = [np.where(meanData >= vmin_threshold, meanData, np.nan)
                     for meanData in meanDatas]
    if symmetric:
        extend = 'both'
        vmax = np.nanmax([np.nanpercentile(np.abs(meanData), 95)
                          for meanData in meanDatas])
        vmin = vmin_threshold if vmin_threshold is not None else -vmax
        norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
        levels=np.linspace(vmin,vmax,numLevels)
    else:
        extend = 'max' if varName=='QCQI' else 'both'
        vmin = vmin_threshold if vmin_threshold is not None else \
        np.nanmin([np.nanpercentile(meanData, 5)for meanData in meanDatas])
        vmax = np.nanmax([np.nanpercentile(meanData, 95) for meanData in meanDatas])
        norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
        levels=np.linspace(vmin,vmax,numLevels)

    #difference colorbar setup
    differenceData = meanDatas[0]-meanDatas[1]
    diff_vmax = np.nanpercentile(np.abs(differenceData), 95)
    diff_vmin = -diff_vmax
    diff_norm = mcolors.TwoSlopeNorm(vmin=diff_vmin, vcenter=0, vmax=diff_vmax)
    diff_levels = np.linspace(diff_vmin, diff_vmax, numLevels)
        
    #Single Plots
    for axis, meanData, title in zip(axes, meanDatas, titles):

        if plotType == 'pcolormesh':
            im = axis.pcolormesh(plotX,plotY,meanData.T,
                                 cmap=cmap,norm=norm)
        elif plotType == 'contourf':
            im = axis.contourf(plotX,plotY,meanData.T,
                               levels=levels,cmap=cmap,norm=norm,extend=extend)

        axis.axhline(0, color='grey', linestyle='--')
        axis.set_ylabel(zLabel); axis.set_title(title)


    #Difference Plot
    if plotType == 'pcolormesh':
        im_diff = axes[2].pcolormesh(plotX,plotY,differenceData.T,
                                     cmap='RdBu_r',norm=diff_norm)
    elif plotType == 'contourf':
        im_diff = axes[2].contourf(plotX,plotY,differenceData.T,
                                   levels=diff_levels,cmap='RdBu_r',norm=diff_norm,extend='both')
        
    axes[2].axhline(0, color='grey', linestyle='--')
    axes[2].set_ylabel(zLabel); axes[2].set_title(f'{titles[0]} - {titles[1]}')
    axes[-1].set_xlabel(xLabel)

    #set colorbars
    fig.colorbar(im,ax=axes[:2],label=label,extend=extend)
    fig.colorbar(im_diff,ax=axes[2],label=fr'$\Delta$ {label}',extend='both')

    #set axis limits
    for axis in axes.ravel():
        axis.set_ylim(z_bins[0],z_bins[-1])
    return fig

# def InterpolateBinnedData(meanData, time_bins):
#     from scipy.interpolate import interp1d
#     t_centers = 0.5 * (time_bins[:-1] + time_bins[1:])  # length = Nt_bins = meanData.shape[0]
#     meanData_filled = meanData.copy()
#     for iz in range(meanData.shape[1]):
#         row = meanData[:, iz]
#         valid = np.isfinite(row)
#         if valid.sum() > 1:
#             f = interp1d(t_centers[valid], row[valid], bounds_error=False, fill_value=np.nan)
#             meanData_filled[:, iz] = f(t_centers)
#     return meanData_filled

def GetMultiplierDictionary():
    multiplierDictionary = {'QV':        1e3,
                            'QCQI':      1e3,
                            'THETA_v':   1,
                            'W':         1,
                            'BUOYANCY2': 1,
                            'z':         1/1e3}

    multiplierDictionary.update({'PROCESSED_E_DivideMassFlux_c': 1, #*testing
                           'PROCESSED_D_DivideMassFlux_c': 1})
    return multiplierDictionary
multiplierDictionary=GetMultiplierDictionary()

def GetCmapDictionary():
    cmapDictionary = {'QV':        'turbo',
                      'QCQI':      'turbo',
                      'THETA_v':   'turbo',
                      'W':         'RdBu_r',
                      'BUOYANCY2': 'RdBu_r',
                      'z':         'viridis'}

    cmapDictionary.update({'PROCESSED_E_DivideMassFlux_c': 'RdBu_r', #*testing
                           'PROCESSED_D_DivideMassFlux_c': 'RdBu_r'})
    return cmapDictionary
cmapDictionary=GetCmapDictionary()

def isVariableSymmetric(varName):
    binaryDictionary = {'QV':       False,
                      'QCQI':       False,
                      'THETA_v':    False,
                      'W':          True,
                      'BUOYANCY2':  True,
                      'z':          False}

    binaryDictionary.update({'PROCESSED_E_DivideMassFlux_c': True, #*testing
                             'PROCESSED_D_DivideMassFlux_c': True})
    return binaryDictionary.get(varName)

In [ ]:
##############################################
#PLOTTING (Z-Relative Histograms Combined)

In [ ]:
#Making Calculations
if not running:
    varNames=['QV','QCQI','THETA_v','W','BUOYANCY2']
    # varNames=['QV'] #*testing
    parcelTypes = ['CL', 'nonCL']
    [meanDataDictionary_before, time_bins_before,z_bins] = MakeCalculations(PrepareData_UpUntilFirstBLAcceleration,priorToAscentArrayDictionary,
                                                              varNames,parcelTypes,parcelDepth='ALL',relative=None)
    [meanDataDictionary_after, time_bins_after,z_bins] = MakeCalculations(PrepareData_FirstBLAccelerationToCloudExit,trajectoriesArrayDictionary,
                                                              varNames,parcelTypes,parcelDepth='ALL',relative=None)

    meanDataDictionary_combined, time_bins_combined = CombineMeanData(meanDataDictionary_before, time_bins_before,
                                                                      meanDataDictionary_after,  time_bins_after)

In [ ]:
#Plotting
if not running:
    for varName in tqdm(varNames):
        meanData_CL = meanDataDictionary_combined[varName][parcelTypes[0]]
        meanData_nonCL = meanDataDictionary_combined[varName][parcelTypes[1]]
        fig = PlotBinnedData(meanData_CL,meanData_nonCL,time_bins_combined,z_bins, 
                             varName=varName,multiplier=multiplierDictionary[varName],
                             cmap=cmapDictionary[varName],symmetric=isVariableSymmetric(varName),
                             label=varName, zLabel='z',titles=parcelTypes,xLabel='time since right before ascent (mins)')

In [ ]:
##############################################
#PLOTTING (CB-Relative Histograms After Initial Acceleration)

In [ ]:
#Making Calculations
if not running:
    varNames=['QV','QCQI','THETA_v','W','BUOYANCY2']
    parcelTypes = ['CL', 'nonCL']
    [meanDataDictionary, time_bins,z_bins] = MakeCalculations(PrepareData_FirstBLAccelerationToCloudExit,trajectoriesArrayDictionary,
                                                              varNames,parcelTypes,parcelDepth='ALL',relative='CB')

In [ ]:
#Plotting
if not running:
    for varName in tqdm(varNames):
        meanData_CL = meanDataDictionary[varName][parcelTypes[0]]
        meanData_nonCL = meanDataDictionary[varName][parcelTypes[1]]
        fig = PlotBinnedData(meanData_CL,meanData_nonCL,time_bins,z_bins, 
                             varName=varName,multiplier=multiplierDictionary[varName],
                             cmap=cmapDictionary[varName],symmetric=isVariableSymmetric(varName), 
                             label=varName, zLabel='z - z_CB (m)',titles=parcelTypes,xLabel='time since right before ascent (mins)')

In [ ]:
##############################################
#PLOTTING (LFC-Relative Histograms After Initial Acceleration)

In [ ]:
#Making Calculations
if not running:
    varNames=['QV','QCQI','THETA_v','W','BUOYANCY2']
    parcelTypes = ['CL', 'nonCL']
    [meanDataDictionary, time_bins,z_bins] = MakeCalculations(PrepareData_FirstBLAccelerationToCloudExit,trajectoriesArrayDictionary,
                                                              varNames,parcelTypes,parcelDepth='ALL',relative='LFC')

In [ ]:
#Plotting
if not running:
    for varName in tqdm(varNames):
        meanData_CL = meanDataDictionary[varName][parcelTypes[0]]
        meanData_nonCL = meanDataDictionary[varName][parcelTypes[1]]
        fig = PlotBinnedData(meanData_CL,meanData_nonCL,time_bins,z_bins,
                             varName=varName, multiplier=multiplierDictionary[varName],
                             cmap=cmapDictionary[varName],symmetric=isVariableSymmetric(varName),
                             label=varName, zLabel='z - z_LFC (m)',titles=parcelTypes,xLabel='time since right before ascent (mins)')

In [ ]:
##############################################
#FUNCTIONS (CB vs CT distribution)

In [ ]:
#Calculation
def CalculateData_CBvsCEDistribution(dataDictionary,varName):
    CB_Values,LFC_Values=[],[]
    CB_Z_Values,LFC_Z_Values,CE_Z_Values,CEminusCB_Z_Values = [],[],[],[]
    for p in tqdm(range(dataDictionary['z'].shape[1])):
        CBIndex = GetTrajectoryCloudBaseTime(dataDictionary,p)
        LFCIndex = GetTrajectoryLFCTime(dataDictionary,p)
        exitIndex = GetTrajectoryLastCloudExitTime(dataDictionary,p)
        if CBIndex and LFCIndex:
            varAtCB=dataDictionary[varName][:,p][CBIndex].item()
            varAtLFC=dataDictionary[varName][:,p][LFCIndex].item()
            
            zAtCB=dataDictionary['z'][:,p][CBIndex].item()
            zAtLFC=dataDictionary['z'][:,p][LFCIndex].item()
            zAtCE=dataDictionary['z'][:,p][exitIndex].item()
            zAtCE_MINUS_zAtCB=zAtCE-zAtCB
            
            CB_Values.append(varAtCB)
            LFC_Values.append(varAtLFC)
            
            CB_Z_Values.append(zAtCB/1e3)
            LFC_Z_Values.append(zAtLFC/1e3)
            CE_Z_Values.append(zAtCE/1e3)
            CEminusCB_Z_Values.append(zAtCE_MINUS_zAtCB/1e3)
    return (CB_Values,LFC_Values,
            CB_Z_Values,LFC_Z_Values,CE_Z_Values,CEminusCB_Z_Values)

def GetTrajectoryLastCloudExitTime(dataDictionary,p):
    array=dataDictionary['z'][:,p]
    exitIndex=FindCloudExitTime(array)
    return exitIndex
def GetTrajectoryCloudBaseTime(dataDictionary,p):
    array=dataDictionary['QCQI'][:,p]
    CBIndex=FindCloudBaseTime(array)
    return CBIndex
def GetTrajectoryLFCTime(dataDictionary,p):
    z=dataDictionary['z'][:,p]
    LFC=dataDictionary['LFC'][:,p]
    LFCIndex=FindLFCTime(z,LFC)
    return LFCIndex
def FindCloudExitTime(array):
    exitIndex = len(array) - 1 - np.argmax(~np.isnan(array[::-1])) - 1
    return exitIndex
def FindCloudBaseTime(array):
    CBIndex = np.where(array > 1e-6)[0]
    if len(CBIndex) == 0:
        return []
    return CBIndex[0]
def FindLFCTime(z,LFC):
    LFCIndex = np.where(z >= LFC)[0]
    if len(LFCIndex) == 0:
        return []
    return LFCIndex[0]

In [ ]:
#Plotting

#Figure One
def MakePlot_SingleLevelDistribution(CB_Values_CL, CB_Values_nonCL, varName, numBins=51,alpha=1,
                                     multiplier=1,atWhichLevel='CB'):
    fig, axis = plt.subplots()

    plotData1=multiplier*np.array(CB_Values_CL)
    plotData2=multiplier*np.array(CB_Values_nonCL)
    axis.hist(plotData1, bins=numBins, color='blue', alpha=alpha, label='CL')
    axis.hist(plotData2, bins=numBins, color='black', alpha=alpha, label='nonCL')

    axis.set_xlabel(f'{varName} at {atWhichLevel}')
    axis.set_ylabel('Count')
    axis.legend()

    return fig

def MakePlot_CBvsLFCScatter(CB_Values_CL, LFC_Values_CL,plotColorData_CL,
                            CB_Values_nonCL, LFC_Values_nonCL,plotColorData_nonCL,
                            varName, multiplier=1,cmap='turbo',colorbarLabel='z at CB (z)'):

    fig, axes = plt.subplots(2, 1, figsize=(7, 8), sharex=True, sharey=True)

    plotData1=multiplier*np.array(CB_Values_CL); plotData2=multiplier*np.array(LFC_Values_CL); plotColorData=np.array(plotColorData_CL)
    scatter1 = axes[0].scatter(plotData1, plotData2,
                               c=plotColorData, cmap=cmap)

    colorbar1 = fig.colorbar(scatter1, ax=axes[0])
    colorbar1.set_label(colorbarLabel)

    axes[0].set_ylabel(f'{varName} at LFC')
    axes[0].set_title('CL')

    plotData1=multiplier*np.array(CB_Values_nonCL); plotData2=multiplier*np.array(LFC_Values_nonCL); plotColorData=np.array(plotColorData_nonCL)
    scatter2 = axes[1].scatter(plotData1, plotData2,
                               c=plotColorData, cmap=cmap)

    colorbar2 = fig.colorbar(scatter2, ax=axes[1])
    colorbar2.set_label(colorbarLabel)

    axes[1].set_xlabel(f'{varName} at CB')
    axes[1].set_ylabel(f'{varName} at LFC')
    axes[1].set_title('nonCL')

    fig.tight_layout()

    return fig

In [ ]:
#Plotting

def MakePlot_CBvsCEDistribution_2Panel(CB_Values_CL, CE_Z_Values_CL, plotColorData_CL,
                                       CB_Values_nonCL, CE_Z_Values_nonCL, plotColorData_nonCL,
                                       varName, multiplier=1, cmap='turbo',
                                       symmetric=True, numLevels=21,
                                       vmin_threshold=None,colorLabel=None):

    fig, axes = plt.subplots(2, 1, figsize=(7, 8), sharey=True, sharex=True)

    yLabel = 'z_CE - z_CB'

    colorData_CL = multiplier * np.array(plotColorData_CL)
    colorData_nonCL = multiplier * np.array(plotColorData_nonCL)
    allColorData = np.concatenate([colorData_CL, colorData_nonCL])

    if vmin_threshold is not None:
        vmin_threshold *= multiplier
        colorData_CL = np.where(colorData_CL >= vmin_threshold, colorData_CL, np.nan)
        colorData_nonCL = np.where(colorData_nonCL >= vmin_threshold, colorData_nonCL, np.nan)

    if symmetric:
        vmax = np.nanpercentile(np.abs(allColorData), 95)
        vmin = -vmax
        # levels = np.linspace(vmin, vmax, numLevels + 1)
        levelsNeg = np.linspace(vmin, 0, numLevels//2 + 1); levelsPos = np.linspace(0, vmax, numLevels//2 + 1)
        levels = np.concatenate([levelsNeg[:-1], levelsPos])
        # norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
        norm = mcolors.BoundaryNorm(boundaries=levels,ncolors=plt.get_cmap(cmap).N,clip=True)
        extend = 'both'
    else:
        vmin = np.nanpercentile(allColorData, 5)
        vmax = np.nanpercentile(allColorData, 95)
        levels = np.linspace(vmin, vmax, numLevels + 1)
        #norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
        norm = mcolors.BoundaryNorm(boundaries=levels,ncolors=plt.get_cmap(cmap).N,clip=True)
        extend = 'both'

    # --- CL ---
    plotData1 = multiplier * np.array(CB_Values_CL)
    plotData2 = np.array(CE_Z_Values_CL)

    scatterPlot1 = axes[0].scatter(plotData1, plotData2,
                                   c=colorData_CL,cmap=cmap,norm=norm)

    axes[0].set_ylabel(yLabel); axes[0].set_title('CL')

    # --- nonCL ---
    plotData1 = multiplier * np.array(CB_Values_nonCL)
    plotData2 = np.array(CE_Z_Values_nonCL)

    scatterPlot2 = axes[1].scatter(plotData1, plotData2,c=colorData_nonCL,cmap=cmap,norm=norm)

    axes[1].set_xlabel(f"{varName} at CB"); axes[1].set_ylabel(yLabel); axes[1].set_title('nonCL')

    colorbar = fig.colorbar(scatterPlot2,ax=axes,ticks=levels,extend=extend)
    if colorLabel is not None: colorbar.set_label(colorLabel)

    # fig.tight_layout()
    return fig

def MakePlot_CBvsCEDistribution_2DHistogram(CB_Values_CL,Z_Values_CL,
                                       CB_Values_nonCL,Z_Values_nonCL,
                                       varName,multiplier=1,cmap='turbo',useLog=False):

    fig, axes = plt.subplots(2, 1,figsize=(7, 8),sharex=True,sharey=True)

    varBinDictionary = GetVarBinDictionary()
    zBins = varBinDictionary['z']
    varBins = varBinDictionary[varName]

    yLabel = 'z_CE - z_CB'

    plotData1=multiplier*np.array(CB_Values_CL); plotData2=np.array(Z_Values_CL) 
    histCL, xEdges, yEdges = np.histogram2d(plotData1,plotData2,bins=[varBins, zBins])

    plotData1=multiplier*np.array(CB_Values_nonCL); plotData2=np.array(Z_Values_nonCL) 
    histNonCL, _, _ = np.histogram2d(plotData1,plotData2,bins=[varBins, zBins])

    maxCount = max(np.nanmax(histCL), np.nanmax(histNonCL))
    if useLog:
        norm = LogNorm(vmin=1, vmax=maxCount) if maxCount>0 else None
    else:
        norm = None

    plot1 = axes[0].pcolormesh(xEdges,yEdges,histCL.T,cmap=cmap,shading='auto',norm=norm)
    colorbar1 = fig.colorbar(plot1, ax=axes[0]); colorbar1.set_label('Counts')

    axes[0].set_ylabel(yLabel); axes[0].set_title('CL');

    plot2 = axes[1].pcolormesh(xEdges,yEdges,histNonCL.T,cmap=cmap,shading='auto',norm=norm)
    colorbar2 = fig.colorbar(plot2, ax=axes[1]); colorbar2.set_label('Counts')

    axes[1].set_xlabel(f'{varName} at CB'); axes[1].set_ylabel(yLabel); axes[1].set_title('nonCL')

    return fig 

def GetVarBinDictionary(numCount=50,zResolution=0.25):
    varBinDictionary = {'QV':        np.linspace(8, 16, numCount),
                        'QCQI':      np.linspace(0, 10, numCount),
                        'THETA_v':   np.linspace(305, 335, numCount),
                        'W':         np.linspace(-5, 30, numCount),
                        'BUOYANCY2': np.linspace(-0.1, 0.15, numCount),
                        'z':         np.arange(-2, 12 + zResolution, zResolution)}
    return varBinDictionary

In [ ]:
##############################################
#PLOTTING

In [ ]:
##############################################
#PLOTTING (CB vs CT distribution)

In [ ]:
#Plotting

if not running:
    parcelType='ALL'
    
    varNames=['QV','QCQI','THETA_v','W','BUOYANCY2','z']
    for varName in varNames:
        #############################
        parcelDepth='CL'
        dataDictionary=trajectoriesArrayDictionary[parcelDepth][parcelType]
        [CB_Values_CL,LFC_Values_CL,
          CB_Z_Values_CL,LFC_Z_Values_CL,CE_Z_Values_CL,CEminusCB_Z_Values_CL] = CalculateData_CBvsCEDistribution(dataDictionary,varName)
        
        parcelDepth='nonCL'
        dataDictionary=trajectoriesArrayDictionary[parcelDepth][parcelType]
        [CB_Values_nonCL,LFC_Values_nonCL,
          CB_Z_Values_nonCL,LFC_Z_Values_nonCL,CE_Z_Values_nonCL,CEminusCB_Z_Values_nonCL] = CalculateData_CBvsCEDistribution(dataDictionary,varName)
    
        #Figure One
        # MakePlot_SingleLevelDistribution(CB_Values_CL, CB_Values_nonCL, 
        #                                  varName,multiplier=multiplierDictionary[varName], atWhichLevel='CB')
        # MakePlot_SingleLevelDistribution(LFC_Values_CL, LFC_Values_nonCL, 
        #                                  varName,multiplier=multiplierDictionary[varName], atWhichLevel='LFC')
        MakePlot_CBvsLFCScatter(CB_Values_CL,LFC_Values_CL,CB_Z_Values_CL,
                                CB_Values_nonCL,LFC_Values_nonCL,CB_Z_Values_nonCL,
                                varName,multiplier=multiplierDictionary[varName],colorbarLabel='z at CB (z)')
        # MakePlot_CBvsLFCScatter(CB_Values_CL,LFC_Values_CL,LFC_Z_Values_CL,
        #                         CB_Values_nonCL,LFC_Values_nonCL,LFC_Z_Values_nonCL,
        #                         varName,multiplier=multiplierDictionary[varName],colorbarLabel='z at LFC (z)')
    
        # #Figure Two
        # fig = MakePlot_CBvsCEDistribution_2Panel(CB_Values_CL,CEminusCB_Z_Values_CL,LFC_Values_CL,
        #                                          CB_Values_nonCL,CEminusCB_Z_Values_nonCL,LFC_Values_nonCL,
        #                                          varName,multiplier=multiplierDictionary[varName],
        #                                          cmap=cmapDictionary[varName],symmetric=isVariableSymmetric(varName),
        #                                          colorLabel=f'{varName} at LFC')
        # # fig = MakePlot_CBvsCEDistribution_2DHistogram(CB_Values_CL,CEminusCB_Z_Values_CL,
        # #                                               CB_Values_nonCL,CEminusCB_Z_Values_nonCL, 
        # #                                               varName,multiplier=multiplierDictionary[varName],
        # #                                               useLog=True)

In [ ]:
########################################
#TESTING

In [ ]:
# #testing parcel trajectory cloudiness & dwdt

# # p=0
# p+=1

# parcelType='CL';parcelDepth='ALL'
# dataDictionary = trajectoriesArrayDictionary[parcelType][parcelDepth]
# z=dataDictionary['z']
# LFC=dataDictionary['LFC']
# qcqi=dataDictionary['QCQI']
# w=dataDictionary['W']; 
# dwdt = np.full_like(w, np.nan); dwdt[1:] = np.diff(w, axis=0) / ModelData.dt
# buoyancy=dataDictionary['BUOYANCY2']
# a1=qcqi[:,p]
# a2=buoyancy[:,p]
# a3=dwdt[:,p]
# b=z[:,p]
# c=LFC[:,p]

# #qcqi
# fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True, gridspec_kw={"wspace":0.03})
# axis=axes[0]
# axis.plot(a1,b)
# axis.set_xlabel('qc+qi (kg/kg)'); axis.set_ylabel('z (m)')
# axis.axvline(1e-6,color='gray',linestyle='--',zorder=-10)
# apply_scientific_notation([axis])

# #w
# axis=axes[1]
# axis.plot(a2,b)
# axis.set_xlabel('B (m/s/s)')#; axis.set_ylabel('z (m)')
# axis.axvline(0,color='gray',linestyle='--',zorder=-10)

# axis=axes[2]
# axis.plot(a3,b)
# axis.set_xlabel(r'$\partial w / \partial t$ (m/s/s)')#; axis.set_ylabel('z (m)')
# axis.axvline(0,color='gray',linestyle='--',zorder=-10)

# #both 
# for axis in axes:
#     axis.axhline(c[(b>c).argmax()],label='lfc',color='orange',linestyle='-',zorder=-5)

In [ ]:
# #Testing Ascent_Condition from Lagrangian_UpdraftTracking
# a=trajectoriesArrayDictionary['CL']['ALL']
# z=a['z'] 
# LFC=a['LFC']
# w=a['W']


# # p=0
# p+=1

# for p in range(z.shape[1]):
    
#     z_p=z[:,p]
#     LFC_p=LFC[:,p]
#     w_p=w[:,p]
#     where=np.where(z_p>LFC_p)[0][0]
#     if np.any(w_p[where:where+2]<=0):
#         print(p)

In [ ]:
# #testing do before and after arrays line up ==> yes
# for p in range(priorToAscentArrayDictionary['CL']['ALL']['z'].shape[1]):
#     a=priorToAscentArrayDictionary['CL']['ALL']['z'][:,p]
#     b=trajectoriesArrayDictionary['CL']['ALL']['z'][:,p]
    
#     where_a=np.where(np.isnan(a))[0][0]-1
#     where_b=np.where(~np.isnan(b))[0][0]
#     if not a[where_a]==b[where_b]:
#         print(p)

In [ ]:
# #testing parcels never in cloud

# #testing
# def CountParcelsNeverInCloud(parcelType):
    
#     array=trajectoriesArrayDictionary[parcelType]['ALL']['QCQI']
#     isNeverCloud = np.all((array <= 1e-6) | np.isnan(array), axis=0)
#     statistic = f"{parcelType}: {np.sum(isNeverCloud)}/{array.shape[1]} = {np.sum(isNeverCloud)/array.shape[1]*100:.2f}%"
#     print(statistic)

# CountParcelsNeverInCloud('CL')
# CountParcelsNeverInCloud('nonCL')

In [ ]:
# # testing parcels min_time of day
# a=trajectoriesArrayDictionary['nonCL']['ALL']['z']
# b=np.isnan(a).all(axis=1)
# c=np.argmin(b)

# times=np.arange(ModelData.Ntime)*ModelData.dt/3600+6
# times[c]

In [ ]:
# #testing overlap between before and after time_bins
# def before(dataDictionary,p=0,varName='W',relative=None):
#     #getting
#     z = dataDictionary['z'][:,p]
#     qcqi = dataDictionary['QCQI'][:,p]
#     LFC = dataDictionary['LFC'][:,p]
#     data = dataDictionary[varName][:,p]
#     time = (np.arange(ModelData.Ntime)*ModelData.dt/3600+6)*60 

#     mask = np.isfinite(z)
    
#     #masking
#     time = time[mask]; time_relative = time-time[-1]

#     return time,time_relative

# # Calculating Functions (First BL Acceleration to Cloud Exit)
# def after(dataDictionary,p=0,varName='W',relative='LFC'):
#     #getting
#     z = dataDictionary['z'][:,p]
#     qcqi = dataDictionary['QCQI'][:,p]
#     LFC = dataDictionary['LFC'][:,p]
#     data = dataDictionary[varName][:,p]
#     time = (np.arange(ModelData.Ntime)*ModelData.dt/3600+6)*60 

#     mask = np.isfinite(z)

#     #masking
#     time = time[mask]; time_relative = time-time[0]; z=z[mask]
    
#     return time,time_relative



# p=2
# [before,before_rel]=before(priorToAscentArrayDictionary['CL']['ALL'],p=p)
# [after,after_rel]=after(trajectoriesArrayDictionary['CL']['ALL'],p=p)

# print(before,'\n',before_rel)
# print('\n')
# print(after,'\n',after_rel)
